# Projects in Math Modelling - Part 1 Sentiment Analysis

# Web Scrapping

https://pypi.org/project/twikit/
https://blog.apify.com/how-to-scrape-tweets-and-more-on-twitter-59330e6fb522/

In [2]:
#install packages
import time
from twikit import Client, ServerError
import json
import pandas as pd
import time
import subprocess
import re

In [3]:

#Function to format names
def format_name(name):
    # Split before an uppercase sequence that follows a lowercase letter
    parts = re.findall(r'[A-Z][a-z]+|(?:[A-Z]+(?=[A-Z]|$))', name)
    
    # Capitalize each part correctly
    formatted_parts = [part.capitalize() for part in parts]
    
    # Join the parts with a space
    return ' '.join(formatted_parts)


In [8]:
#Load starter list

# Load the JSON data from the file
with open('final_results.json', 'r') as json_file:
    results = json.load(json_file)

# Extract the list of athlete names
athletes = [entry['Name'] for entry in results if 'Name' in entry]

#Take competitor column and format it into a string list to use later on as keywords
#competitor = athletes["Competitor"].astype(str).tolist()
competitor = [format_name(athlete) for athlete in athletes]


['Jakob Ingebrigtsen', 'Ronald Kwemoi', 'Grant Fisher', 'Dominic Lokinyomo Lobalu', 'Hagos Gebrhiwet', 'Biniam Mehary', 'Edwin Kurgat', 'Isaac Kimeli', 'Graham Blanks', 'Jacob Krop', 'John Heymans', 'Yann Schrub', 'Mike Foppen', 'Addisu Yihune', 'Thierry Ndikumwenayo', 'Hugo Hay', 'Narve Gilje Nordas', 'Stewart Mcsweyn', 'Dawit Seare', 'Oscar Chelimo', 'George Mills', 'Thomas Fafard']


In [20]:
# Start VPN connection
path="C:\Program Files\Proton\VPN\ProtonVPN.Launcher.exe"
subprocess.run([path, 'connect', 'Ireland'])


CompletedProcess(args=['C:\\Program Files\\Proton\\VPN\\ProtonVPN.Launcher.exe', 'connect', 'Ireland'], returncode=0)

In [24]:
#initialize client
client = Client()

In [22]:
#Pull Twitter login information
with open("twitter_login.json") as infile:
    json_obj = json.load(infile)
    username =json_obj["E-Mail4"]
    password =json_obj["Password4"]

In [27]:
#Loggin into account on Twitter
retries = 3
for attempt in range(retries):
    try:
        client.login(auth_info_1=username, password=password)
        client.save_cookies('cookies.json')
        client.load_cookies(path='cookies.json')
        break  # Success, exit the retry loop
    except ServerError as e:
        if attempt < retries - 1:
            time.sleep(5)  # Wait before retrying
        else:
            raise  # Re-raise the error if out of retries


Om je account te beschermen tegen verdachte activiteiten, hebben we een bevestigingscode verzonden naar be*********@p*****.**. Voer deze hieronder in om in te loggen.	


In [28]:
# Function to handle rate limit (only limited amount of requests per 15 minutes allowed)
def handle_rate_limit():
    print("Rate limit hit. Sleeping for 15 minutes.")
    time.sleep(15 * 60)  # Sleep for 15 minutes

In [64]:
#webscape the tweets
tweets_to_store = []
for name in competitor:
    query = f'"{name}"'
    
    while True:
        try:
            tweets = client.search_tweet(query, 'Mixed')
            break  # Exit the loop if request was successful
        except Exception as e:
            if "Rate limit exceeded" in str(e):
                handle_rate_limit()
            else:
                print(f"An error occurred: {e}")
                break


    if tweets is None:
        print(f"No tweets found for athlete: {name}")
        continue
    
    for tweet in tweets:
        tweets_to_store.append({
            'created_at': tweet.created_at,
            'username': tweet.user.name,
            'Text': tweet.text,
            'Competitor': name
        })
    
    # Sleep for a short interval to avoid hitting the rate limit
    time.sleep(5)

An error occurred: status: 400, message: "{"errors":[{"message":"Variable 'product' has an invalid value: Invalid input for enum 'SearchProductInput'. No value found for name 'Mixed'","extensions":{"name":"ValidationError","source":"Client","code":366,"kind":"Validation","tracing":{"trace_id":"0f7acd3b189798b3"}},"code":366,"kind":"Validation","name":"ValidationError","source":"Client","tracing":{"trace_id":"0f7acd3b189798b3"}}]}"
An error occurred: status: 400, message: "{"errors":[{"message":"Variable 'product' has an invalid value: Invalid input for enum 'SearchProductInput'. No value found for name 'Mixed'","extensions":{"name":"ValidationError","source":"Client","code":366,"kind":"Validation","tracing":{"trace_id":"87e9934b3513aae5"}},"code":366,"kind":"Validation","name":"ValidationError","source":"Client","tracing":{"trace_id":"87e9934b3513aae5"}}]}"


KeyboardInterrupt: 

In [62]:
# Save the tweet details to a JSON file
with open('tweets_final_relevance.json', 'w') as json_file:
    json.dump(tweets_to_store, json_file, indent=4)




In [ ]:
# Disconnect VPN
subprocess.run([path, 'disconnect'])